UNIVERSIDAD CENTRAL DE VENEZUELA   
FACULTAD DE CIENCIAS ECONÓMICAS Y SOCIALES   
ESCUELA DE ESTADÍSTICA Y CIENCIAS ACTUARIALES   
DEPARTAMENTO DE ESTADÍSTICA Y PROBABILIDAD   
CÁTEDRA DE DEMOGRAFÍA   
SEMESTRE I-2026   

# Análisis Demográfico del Estado Lara

Autor: Giovanni Fermin    
C.I: 31126857

Lara es un estado de Venezuela que pertenece a la Región Centro Occidental del país. Está conformado por un total de nueve municipios: Crespo, Iribarren, Jiménez, Morán, Palavecino, Torres, Andrés Eloy Blanco, Simón Planas y Urdaneta.
Su principal riqueza radica en la agricultura, en la ganadería, artesanía y maderería. Cuenta con una superficie de 19.800 kmˆ2.   
Lara registró un total de 1.774.867 habitantes en el censo de 2011, lo cual representa aproximadamente el 6.5% de la población nacional. Dicho censo arrojó un crecimiento poblacional del 1.3% con respecto al censo del 2001. Además a través de la pirámide poblacional del mismo año se mostró que la población de edad de trabajar superaba a la población en edades de dependencia.  
Al ubicarse en el centro occidente del país, Lara es un estado clave del territorio venezolano que sirve como puente entre la región Centro Occidental y la región de Los Llanos; por lo que se concentra una actividad económica importante, específicamente en el eje Barquisimeto-Cabudare, atrayendo mano de obra y desplazando la economía hacia los sectores económicos propios del estado que fueron anteriormente nombrados.

A investigar, tablas por sexo y rango de edades, realizar analisis de tendencias con graficos: 
- Población
- A partir de 10 años población con condición de alfabetismo
- a partir de 12 años por situacion conyugal
- a partir de 10 años por situación en la fuerza de trabajo
- a partir de 10 años por ocupación del nivel de instrucción (primaria, secundaria y universitaria)
- Relaciones de dependencia económica, población ocupada, rama de acticidad económica, etc.
- a partir de 10 años por actividad económica 
- Índice de masculinidad 

Comparar las tasas adquiridas con datos del pasado del mismo estado o comparado con Venezuela (Siempre por sexo y grupo de edad)

Excluir a los no declarados en el análisis de situación conyugal

In [67]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Funciones de Estructuración de Archivos Excel

In [68]:
def cargar_censo_2011(filepath: str) -> pd.DataFrame:
    df = pd.read_excel(filepath, header=2)
    df = df.dropna(how='all', axis=1).dropna(how='all', axis=0)
    df.columns = [str(c).strip() for c in df.columns]
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
    df = df.dropna(subset=['Entidad Federal']).reset_index(drop=True)
    return df

def cargar_matrimonios(filepath: str) -> pd.DataFrame:
    df_raw = pd.read_excel(filepath, header=None)
    years = [int(y) if isinstance(y, (int, float)) and pd.notnull(y) else str(y).strip() for y in df_raw.iloc[2, 1:]]
    cols = ['Entidad Federal'] + years
    df = df_raw.iloc[3:].copy()
    df.columns = cols
    df = df.dropna(subset=['Entidad Federal'])
    df['Entidad Federal'] = df['Entidad Federal'].astype(str).str.strip()
    return df.reset_index(drop=True)

def cargar_fuerza_trabajo(filepath: str) -> pd.DataFrame:
    df_raw = pd.read_excel(filepath, header=None)
    years = [int(y) if isinstance(y, (int, float)) and pd.notnull(y) else str(y).strip() for y in df_raw.iloc[1, 5:]]
    rows = []
    for idx in range(2, len(df_raw)):
        row = df_raw.iloc[idx]
        cat_levels = [str(row[i]).strip() for i in range(5) if pd.notnull(row[i])]
        concept = ' - '.join(cat_levels)
        values = row[5:].values
        if concept and any(pd.notnull(v) for v in values):
            rows.append([concept] + list(values))
    df = pd.DataFrame(rows, columns=['Indicador'] + years)
    return df

def cargar_viviendas_hogares_personas(filepath: str) -> pd.DataFrame:
    df_raw = pd.read_excel(filepath, header=None)
    df_raw = df_raw.dropna(how='all', axis=1).dropna(how='all', axis=0)
    df_raw.columns = ['Categoria', 'Indicador', 'Valor']
    df_raw['Categoria'] = df_raw['Categoria'].ffill()
    df_raw = df_raw.dropna(subset=['Indicador', 'Valor'])
    df_raw['Categoria'] = df_raw['Categoria'].astype(str).str.strip()
    df_raw['Indicador'] = df_raw['Indicador'].astype(str).str.strip()
    return df_raw.reset_index(drop=True)

### Carga de DataFrames 

In [69]:
df_poblacion_sexo = cargar_censo_2011("Poblacion-Sexo-Censo-2011.xlsx")
df_poblacion_grupos_edad = cargar_censo_2011("Poblacion-Grupos-Edad-Censo-2011.xlsx")
df_poblacion_nacida_venezuela = cargar_censo_2011("Poblacion-Nacida-en-Venezuela-Censo-2011.xlsx")
df_por_entidad_matrimonios = cargar_matrimonios("POR-ENTIDAD-Matrimonios.xls")
df_trabajo = cargar_fuerza_trabajo("TRABAJO.xlsx")
df_viviendas_hogares_personas = cargar_viviendas_hogares_personas("VIVIENDAS-HOGARES-Y-PERSONAS-CENSO-2011.xlsx")
df_lara = pd.read_excel("Lara.xls")
df_alfabetismo_edad = pd.read_excel("alfabetismo.xlsx", sheet_name="Por_Edad")
df_alfabetismo_sexo = pd.read_excel("alfabetismo.xlsx", sheet_name="Por_Sexo")
df_alfabetismo_edad_sexo = pd.read_excel("alfabetismo.xlsx", sheet_name="Por_Edad_y_Sexo")
df_conyugal_hombres = pd.read_excel("situacion_conyugal.xlsx", sheet_name="Hombres")
df_conyugal_mujeres = pd.read_excel("situacion_conyugal.xlsx", sheet_name="Mujeres")
df_conyugal_total = pd.read_excel("situacion_conyugal.xlsx", sheet_name="Total")
df_instruccion_edad = pd.read_excel("nivel_instruccion.xlsx", sheet_name="Por_Edad")
df_instruccion_edad_10_mas = pd.read_excel("nivel_instruccion.xlsx", sheet_name="Por_Edad_10_mas")
df_instruccion_sexo = pd.read_excel("nivel_instruccion.xlsx", sheet_name="Por_Sexo")
df_raw_ft = pd.read_excel("TRABAJO.xlsx", header=None)
col_2011 = next((c for c in range(df_raw_ft.shape[1]) if "2011" in str(df_raw_ft.iloc[1, c])), 17)

/Users/firewew/Documents/GitHub/An-lisis_demogr-fico_del_estado_Lara_VEN/Demografia/lib/python3.14/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/Users/firewew/Documents/GitHub/An-lisis_demogr-fico_del_estado_Lara_VEN/Demografia/lib/python3.14/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/Users/firewew/Documents/GitHub/An-lisis_demogr-fico_del_estado_Lara_VEN/Demografia/lib/python3.14/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


### Parroquias Empadronadas

In [70]:
df_lara_sexo = df_poblacion_sexo[df_poblacion_sexo['Entidad Federal'].str.contains('Lara', na=False)].reset_index(drop=True)
df_lara_grupos_edad = df_poblacion_grupos_edad[df_poblacion_grupos_edad['Entidad Federal'].str.contains('Lara', na=False)].reset_index(drop=True)
df_lara_nacida_venezuela = df_poblacion_nacida_venezuela[df_poblacion_nacida_venezuela['Entidad Federal'].str.contains('Lara', na=False)].reset_index(drop=True)
df_lara_matrimonios = df_por_entidad_matrimonios[df_por_entidad_matrimonios['Entidad Federal'].str.contains('Lara', na=False)].reset_index(drop=True)

print(f"Parroquias empadronadas en Lara: {len(df_lara_sexo)}")

Parroquias empadronadas en Lara: 57


### Análisis de Población por Sexo

In [71]:
totales_sexo = pd.DataFrame(
{
    'Hombres': [df_lara_sexo['Hombre'].sum(), df_lara_sexo['Hombre'].sum()/df_lara_sexo['Total'].sum()*100],
    'Mujeres': [df_lara_sexo['Mujer'].sum(), df_lara_sexo['Mujer'].sum()/df_lara_sexo['Total'].sum()*100],
    'Total': [df_lara_sexo['Total'].sum(), df_lara_sexo['Total'].sum()/df_lara_sexo['Total'].sum()*100]
},
index=['Cantidad', 'Porcentaje'])
print(totales_sexo)
df_lara_sexo[['Municipio', 'Parroquia', 'Total', 'Hombre', 'Mujer']].head(10)

                  Hombres        Mujeres      Total
Cantidad    883775.000000  891092.000000  1774867.0
Porcentaje      49.793872      50.206128      100.0


,Municipio,Parroquia,Total,Hombre,Mujer
0,"Lara, Andrés Eloy Blanco",Pío Tamayo,30342.0,15413.0,14929.0
1,"Lara, Andrés Eloy Blanco",Quebrada Honda de Guache,7460.0,4177.0,3283.0
2,"Lara, Andrés Eloy Blanco",Yacambú,9443.0,5325.0,4118.0
3,"Lara, Crespo",Fréitez,32341.0,16396.0,15945.0
4,"Lara, Crespo",José María Blanco,17617.0,9035.0,8582.0
5,"Lara, Iribarren",Catedral,132302.0,63308.0,68994.0
6,"Lara, Iribarren",Concepción,115679.0,55895.0,59784.0
7,"Lara, Iribarren",El Cují,55931.0,29580.0,26351.0
8,"Lara, Iribarren",Juan de Villegas,344196.0,169479.0,174717.0
9,"Lara, Iribarren",Santa Rosa,75755.0,36174.0,39581.0


In [72]:
df_edad_sexo = pd.merge(
    left=df_lara_sexo,
    right=df_lara_grupos_edad,
    on=['Código UBIGEO', 'Entidad Federal', 'Municipio', 'Parroquia', 'Total'],
    suffixes=('_sexo', '_edad'),
).copy()

edades = ['De 10 a 14 años', 'De 15 a 19 años', 'De 20 a 24 años','De 25 a 29 años', 'De 30 a 34 años', 'De 35 a 39 años','De 40 a 44 años', 'De 45 a 49 años', 'De 50 a 54 años','De 55 a 59 años', 'De 60 a 64 años', 'De 65 a 69 años','De 70 a 74 años', 'De 75 a 79 años', 'De 80 a 84 años','De 85 a 89 años', 'De 90 a 94 años', '95 años y Más']
totales_por_edad = df_edad_sexo[edades].sum()
prop_hombres = df_lara_sexo['Hombre'].sum() / df_lara_sexo['Total'].sum()
prop_mujeres = df_lara_sexo['Mujer'].sum() / df_lara_sexo['Total'].sum()
Tabla_edad_sexo = pd.DataFrame(
    {
        'Hombre': (totales_por_edad * prop_hombres).round().astype(int),
        'Mujer': (totales_por_edad * prop_mujeres).round().astype(int),
        'Total': totales_por_edad.astype(int),
    },
    index=edades,
)
Tabla_edad_sexo.loc['Total'] = Tabla_edad_sexo.sum()
Tabla_edad_sexo

,Hombre,Mujer,Total
De 10 a 14 años,83102,83790,166892
De 15 a 19 años,84866,85569,170435
De 20 a 24 años,80032,80695,160727
De 25 a 29 años,76479,77113,153592
De 30 a 34 años,73343,73951,147294
De 35 a 39 años,61395,61904,123299
De 40 a 44 años,57443,57918,115361
De 45 a 49 años,51004,51426,102430
De 50 a 54 años,44454,44822,89276
De 55 a 59 años,35640,35936,71576


In [73]:
print(f'El índice de masculinidad del estado Lara corresponde a IMˆ2011 = : {round(Tabla_edad_sexo['Hombre']['Total'] / Tabla_edad_sexo['Mujer']['Total'] *100,2)}\n Indicando que hay aproximadamente 99 hombres por cada 100 mujeres')

El índice de masculinidad del estado Lara corresponde a IMˆ2011 = : 99.18
 Indicando que hay aproximadamente 99 hombres por cada 100 mujeres


In [74]:
indice_masc_edad = pd.DataFrame({
    'IMˆ2011': round(Tabla_edad_sexo['Hombre']/Tabla_edad_sexo['Mujer']*100,3)
})
indice_masc_edad

,IMˆ2011
De 10 a 14 años,99.179
De 15 a 19 años,99.178
De 20 a 24 años,99.178
De 25 a 29 años,99.178
De 30 a 34 años,99.178
De 35 a 39 años,99.178
De 40 a 44 años,99.180
De 45 a 49 años,99.179
De 50 a 54 años,99.179
De 55 a 59 años,99.176


Estos resultados demuestran que el Índice de Masculinidad por grupos de edad del Estado Lara se aproxima a 99, es decir, que por cada 100 mujeres hay 99 hombres, lo que concuerda con. el Índice de masculinidad total

### Análisis de Condición de alfabetismo (a partir de 10 años)

In [75]:
Tabla_alfabetismo_edad_sexo = df_alfabetismo_edad_sexo.set_index("Grupo de edad")
Tabla_alfabetismo_edad_sexo


,Hombre Alfabeta,Hombre Analfabeta,Hombre Total,Mujer Alfabeta,Mujer Analfabeta,Mujer Total,Total Alfabeta,Total Analfabeta,Total General
Grupo de edad,,,,,,,,,
De 10 a 14 años,80612,1869,82481,82664,1747,84411,163276,3616,166892
De 15 a 19 años,82410,1823,84233,84500,1702,86202,166910,3525,170435
De 20 a 24 años,77496,1939,79435,79482,1810,81292,156978,3749,160727
De 25 a 29 años,73978,1930,75908,75882,1802,77684,149860,3732,153592
De 30 a 34 años,70416,2380,72796,72275,2223,74498,142691,4603,147294
De 35 a 39 años,58464,2473,60937,60049,2313,62362,118513,4786,123299
De 40 a 44 años,54428,2586,57014,55928,2419,58347,110356,5005,115361
De 45 a 49 años,47574,3049,50623,48951,2856,51807,96525,5905,102430
De 50 a 54 años,40694,3428,44122,41938,3216,45154,82632,6644,89276


In [76]:
Tabla_alfabetismo_sexo = df_alfabetismo_sexo.set_index("Sexo")
Tabla_alfabetismo_sexo

,Alfabeta,Analfabeta,Total
Sexo,,,
Hombre,681448,41865,723313
Mujer,700754,39475,740229
Total,1382202,81340,1463542


In [77]:
Tanalf = pd.DataFrame({
    'T_anlfˆ2011': (Tabla_alfabetismo_sexo['Analfabeta']/Tabla_alfabetismo_sexo['Total']*100)
})
Tanalf

,T_anlfˆ2011
Sexo,
Hombre,5.787951
Mujer,5.332809
Total,5.557750


In [78]:
Tanalf_edad_sexo = pd.DataFrame({
    'Tanalf_por_hombres': Tabla_alfabetismo_edad_sexo['Hombre Analfabeta'] / Tabla_alfabetismo_edad_sexo['Total General']*100,
    'Tanalf_por_mujeres': Tabla_alfabetismo_edad_sexo['Mujer Analfabeta'] / Tabla_alfabetismo_edad_sexo['Total General']*100
})
Tanalf_edad_sexo

,Tanalf_por_hombres,Tanalf_por_mujeres
Grupo de edad,,
De 10 a 14 años,1.119886,1.046785
De 15 a 19 años,1.069616,0.998621
De 20 a 24 años,1.206393,1.126133
De 25 a 29 años,1.256576,1.173238
De 30 a 34 años,1.615816,1.509226
De 35 a 39 años,2.005693,1.875928
De 40 a 44 años,2.241659,2.096896
De 45 a 49 años,2.976667,2.788246
De 50 a 54 años,3.839778,3.602312


La Tasa de analfabetismo analizada por sexo y grupo de edad, expresa una clara evolución de educación en Lara, se observa un decrecimiento importante en el analfabetismo tanto en hombres como en mujeres a medida que va disminuyendo la edad. Indicando de esta manera que la educación ha sido generalizada tanto en hombres como en mujeres al pasar el tiempo, permitiendo obtener una población larense más educada

### Análisis de situación Conyugal 

In [79]:
Tabla_conyugal_total = df_conyugal_total.set_index("Grupo de edad")
print('Situación Conyugal en el estado Lara:')
Tabla_conyugal_total

Situación Conyugal en el estado Lara:


,Unido(a),Casado(a),Soltero(a),Separado(a) de unión o matrimonio,Divorciado(a),Viudo(a) de unión o matrimonio,Total
Grupo de edad,,,,,,,
De 12 a 14 años,511,91,165663,21,0,2,166288
De 15 a 19 años,13859,1622,152930,445,32,54,168942
De 20 a 24 años,42345,10092,104768,1734,134,194,159267
De 25 a 29 años,56442,25446,66749,2994,522,321,152474
De 30 a 34 años,56552,36758,46822,4227,1510,495,146364
De 35 a 39 años,45127,37361,32716,4515,2332,769,122820
De 40 a 44 años,38459,38656,28360,4951,3301,1229,114956
De 45 a 49 años,30743,36717,23842,5112,3799,1880,102093
De 50 a 54 años,23385,33276,20343,4819,4294,2872,88989


In [80]:
Tabla_conyugal_hombres = df_conyugal_hombres.set_index("Grupo de edad")
print('Situación Conyugal por Hombres:')
Tabla_conyugal_hombres

Situación Conyugal por Hombres:


,Unido(a),Casado(a),Soltero(a),Separado(a) de unión o matrimonio,Divorciado(a),Viudo(a) de unión o matrimonio,Total
Grupo de edad,,,,,,,
De 12 a 14 años,65,0,86152,4,0,0,86221
De 15 a 19 años,3194,400,82630,52,19,11,86306
De 20 a 24 años,17464,3338,58906,378,44,27,80157
De 25 a 29 años,27049,10739,36625,798,171,36,75418
De 30 a 34 años,28929,17113,25191,1413,532,85,73263
De 35 a 39 años,23129,18192,16603,1510,799,140,60373
De 40 a 44 años,20035,19038,14251,1744,1123,276,56467
De 45 a 49 años,16212,18088,11429,1744,1266,352,49091
De 50 a 54 años,12989,16848,9456,1714,1330,566,42903


In [81]:
Tabla_conyugal_mujeres = df_conyugal_mujeres.set_index("Grupo de edad")
print('Situación Conyugal por Mujeres:')
Tabla_conyugal_mujeres

Situación Conyugal por Mujeres:


,Unido(a),Casado(a),Soltero(a),Separado(a) de unión o matrimonio,Divorciado(a),Viudo(a) de unión o matrimonio,Total
Grupo de edad,,,,,,,
De 12 a 14 años,446,91,79511,17,0,2,80067
De 15 a 19 años,10665,1222,70300,393,13,43,82636
De 20 a 24 años,24881,6754,45862,1356,90,167,79110
De 25 a 29 años,29393,14707,30124,2196,351,285,77056
De 30 a 34 años,27623,19645,21631,2814,978,410,73101
De 35 a 39 años,21998,19169,16113,3005,1533,629,62447
De 40 a 44 años,18424,19618,14109,3207,2178,953,58489
De 45 a 49 años,14531,18629,12413,3368,2533,1528,53002
De 50 a 54 años,10396,16428,10887,3105,2964,2306,46086


Población a partir de 12 años según la captación censal de situación conyugal. Se ignoraron 7.929 individuos según el reporte del INE.

### Análisis de Nivel de Instrucción

In [82]:
Tabla_instruccion_10_mas = df_instruccion_edad_10_mas.set_index("Grupo de edad")
Tabla_instruccion_10_mas

,Ninguno,Primaria (1-6),"Secundaria (1-5),(6)",Técnico Superior,Universitario,No sabe,Total
Grupo de edad,,,,,,,
De 10 a 14 años,1796,113306,49647,0,0,2143,166892
De 15 a 19 años,2589,23745,124370,2470,15381,1880,170435
De 20 a 24 años,2984,23270,74343,11067,46456,2607,160727
De 25 a 29 años,3221,28544,68506,12880,37864,2577,153592
De 30 a 34 años,4301,33748,64827,10968,30626,2824,147294
De 35 a 39 años,4536,32839,53522,7981,22021,2400,123299
De 40 a 44 años,4851,33064,49843,6466,18641,2496,115361
De 45 a 49 años,5716,33446,41023,4124,15760,2361,102430
De 50 a 54 años,6577,32494,31789,2830,13029,2557,89276


In [83]:
Tabla_instruccion_sexo = df_instruccion_sexo.set_index("Sexo")
Tabla_instruccion_sexo

,No sabe,Ninguno,Inicial (Preescolar),Primaria (1-6),"Secundaria (1-5),(6)",Técnico Superior,Universitario,Total
Sexo,,,,,,,,
Hombre,21829,63636,36366,302359,302076,26680,83676,836622
Mujer,18573,59343,34394,250147,307486,36419,140224,846586
Total,40402,122979,70760,552506,609562,63099,223900,1683208


Población a partir de 10 años por nivel de instrucción. Se ignoraron 91.659 individuos según el reporte del INE.

### Fuerza de Trabajo y Actividad Económica

In [84]:


# Cuadro 04: Situación en la fuerza de trabajo por grupos de edad (2011)
grupos_edad = ["15 - 24", "25 - 44", "45 - 64", "65 Y MAS"]

rows_activa = {df_raw_ft.iloc[r, 3].strip(): int(df_raw_ft.iloc[r, col_2011]) for r in [98, 100, 102, 104]}
rows_ocupada = {df_raw_ft.iloc[r, 4].strip(): int(df_raw_ft.iloc[r, col_2011]) for r in [108, 110, 112, 114]}
rows_desocupada = {df_raw_ft.iloc[r, 4].strip(): int(df_raw_ft.iloc[r, col_2011]) for r in [118, 120, 122, 124]}
rows_inactiva = {df_raw_ft.iloc[r, 3].strip(): int(df_raw_ft.iloc[r, col_2011]) for r in [128, 130, 132, 134]}

Tabla_fuerza_trabajo_edad = pd.DataFrame({
    "Población Activa": rows_activa,
    "Ocupados": rows_ocupada,
    "Desocupados": rows_desocupada,
    "Inactivos": rows_inactiva
}, index=grupos_edad)
Tabla_fuerza_trabajo_edad.index.name = "Grupo de edad"
Tabla_fuerza_trabajo_edad["Total (PET 15+)"] = (
    Tabla_fuerza_trabajo_edad["Población Activa"] + Tabla_fuerza_trabajo_edad["Inactivos"]
)
Tabla_fuerza_trabajo_edad.loc["Total (15 y más)"] = Tabla_fuerza_trabajo_edad.sum()

# Cuadro 05: Rama de actividad económica de la población activa (2011)
branch_rows = [
    (162, "Agricultura, Pecuaria y Caza"),
    (169, "Industria Manufacturera"),
    (176, "Construcción"),
    (183, "Comercio, Restaurantes y Hoteles"),
    (190, "Transporte, Almacenamiento y Comunicaciones"),
    (197, "Establecimientos Financieros y Seguros"),
    (204, "Servicios Comunales, Sociales y Personales"),
    (211, "Explotación de Minas y Canteras"),
    (218, "Electricidad, Gas y Agua"),
    (225, "No Declaradas / No Especificadas")
]

data_ramas = []
for start_r, name in branch_rows:
    activa = int(df_raw_ft.iloc[start_r + 1, col_2011])
    ocupada = int(df_raw_ft.iloc[start_r + 3, col_2011])
    desocupada = int(df_raw_ft.iloc[start_r + 5, col_2011])
    data_ramas.append({
        "Rama de Actividad": name,
        "Población Activa": activa,
        "Ocupados": ocupada,
        "Desocupados": desocupada
    })

Tabla_actividad_economica = pd.DataFrame(data_ramas).set_index("Rama de Actividad")
Tabla_actividad_economica.loc["Total PEA"] = Tabla_actividad_economica.sum()

Tabla_fuerza_trabajo_edad


# Este cuadro contiene las estadisticas nacionales, no se encuentra la opción de Lara, preguntar a la profesora
# Los cuadros faltantes son: Actividad Económica y Fuerza de Trabajo

,Población Activa,Ocupados,Desocupados,Inactivos,Total (PET 15+)
Grupo de edad,,,,,
15 - 24,2172150,1771279,400871,3129740,5301890
25 - 44,6895685,6349060,546624,1515927,8411612
45 - 64,3652478,3467606,184872,1484397,5136875
65 Y MAS,442318,417014,25303,1259350,1701668
Total (15 y más),13162631,12004959,1157670,7389414,20552045
